In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

train.shape, test.shape

((1460, 81), (1459, 80))

In [4]:
X = train.drop("SalePrice", axis=1)
y = train["SalePrice"]

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=["str"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(random_state=42, n_jobs=-1))
])

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
# Define hyperparameter search space:
param_distributions = {
    "regressor__n_estimators": [100, 200, 500, 1000],
    "regressor__max_depth":    [None, 10, 20, 30, 50],
    "regressor__max_features": ["sqrt", "log2", 1.0],
    "regressor__min_samples_split": [2,5,10],
    "regressor__min_samples_leaf": [1,2,4]
}

In [8]:
# Run RandomizedSearchCV

import time
start = time.time()

search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_distributions,
    n_iter=30,
    cv=5,
    scoring="neg_mean_absolute_error",
    random_state=42,
    n_jobs=-1,
    verbose=2
)

search.fit(X_train, y_train)

print(f"\n[DONE] Took {time.time() - start:.1f} seconds")
print(f"Best CV MAE: ${-search.best_score_:,.0f}")
print(f"Best params:")
for k, v in search.best_params_.items():
    print(f"  {k}: {v}")

Fitting 5 folds for each of 30 candidates, totalling 150 fits

[DONE] Took 62.0 seconds
Best CV MAE: $18,004
Best params:
  regressor__n_estimators: 500
  regressor__min_samples_split: 2
  regressor__min_samples_leaf: 1
  regressor__max_features: sqrt
  regressor__max_depth: None


In [10]:
# RandomizedSearchCV automatically refits the best model on all training data
best_model = search.best_estimator_

# Score on the held-out validation set
y_pred = best_model.predict(X_val)
val_mae = mean_absolute_error(y_val, y_pred)
val_r2 = r2_score(y_val, y_pred)

print(f"Tuned model validation MAE: ${val_mae:,.0f}")
print(f"Tuned model validation R²:  {val_r2:.2f}")

Tuned model validation MAE: $18,452
Tuned model validation R²:  0.86
